
## Introduction

This notebook demonstrates training a Wasserstein GAN with Gradient Penalty (WGAN-GP) on the MNIST dataset using TensorFlow 2.x. The training utilities are shared with `train_wgan_gp_tf.py` so the notebook and command-line workflow stay consistent. Feel free to tweak hyperparameters such as the learning rate, critic iterations, gradient-penalty strength, and model depth to study their impact on sample quality and training stability.


In [ ]:

# === Environment & Utilities ===
import json
import math
import os
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from train_wgan_gp_tf import (
    make_mnist_dataset,
    build_generator,
    build_discriminator,
    WGANGPTrainer,
)


In [ ]:

# === Dataloaders ===
def get_mnist_dataset(batch_size=64, shuffle_buffer=60000):
    """Prepare the MNIST dataset scaled to [-1, 1]."""
    return make_mnist_dataset(batch_size=batch_size, buffer_size=shuffle_buffer)



## Model Definitions
We build lightweight convolutional generator and critic networks with TensorFlow/Keras layers. Adjust the base channel count to control the model capacity.


In [ ]:

# === Models ===
def create_models(latent_dim=128, base_channels=96):
    generator = build_generator(latent_dim=latent_dim, base_channels=base_channels)
    discriminator = build_discriminator(base_channels=base_channels)
    return generator, discriminator


In [ ]:

# === Training Utils ===
def preview_batch(dataset, n=16):
    batch = next(iter(dataset.unbatch().batch(n)))
    images = (batch.numpy() + 1.0) * 0.5
    rows = int(math.sqrt(n))
    cols = rows
    fig, axes = plt.subplots(rows, cols, figsize=(cols, rows))
    for ax, img in zip(axes.flatten(), images):
        ax.imshow(img.squeeze(), cmap='gray')
        ax.axis('off')
    plt.show()


def save_image_grid(generator, latent_dim, grid_size, out_path):
    noise = tf.random.normal([grid_size ** 2, latent_dim])
    preds = generator(noise, training=False)
    preds = (preds + 1.0) * 0.5
    preds = tf.clip_by_value(preds, 0.0, 1.0)
    canvas = np.zeros((28 * grid_size, 28 * grid_size))
    imgs = preds.numpy().reshape(grid_size ** 2, 28, 28)
    for i in range(grid_size):
        for j in range(grid_size):
            canvas[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = imgs[i * grid_size + j]
    plt.figure(figsize=(grid_size, grid_size))
    plt.imshow(canvas, cmap='gray')
    plt.axis('off')
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
    plt.close()


In [ ]:

# === Trainer (notebook version) ===
class NotebookTrainer:
    def __init__(
        self,
        generator,
        discriminator,
        dataset,
        batch_size,
        latent_dim=128,
        n_critic=3,
        gp_weight=10.0,
        lr=2e-4,
        beta1=0.0,
        beta2=0.99,
        output_dir="outputs/notebook_run",
        sample_every=5,
        sample_grid=6,
        max_batches=0,
        eager=False,
    ):
        if eager:
            tf.config.run_functions_eagerly(True)
        self.dataset = dataset
        self.batch_size = batch_size
        self.latent_dim = latent_dim
        self.sample_every = sample_every
        self.sample_grid = sample_grid
        self.max_batches = max_batches
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        g_opt = tf.keras.optimizers.Adam(lr, beta_1=beta1, beta_2=beta2)
        d_opt = tf.keras.optimizers.Adam(lr, beta_1=beta1, beta_2=beta2)

        self.trainer = WGANGPTrainer(
            latent_dim,
            batch_size,
            n_critic,
            gp_weight,
            generator,
            discriminator,
            g_opt,
            d_opt,
        )
        self.history = {"g_loss": [], "d_loss": [], "wasserstein": []}
        self.config = dict(
            latent_dim=latent_dim,
            n_critic=n_critic,
            gp_weight=gp_weight,
            lr=lr,
            beta1=beta1,
            beta2=beta2,
            batch_size=batch_size,
            sample_every=sample_every,
            sample_grid=sample_grid,
            max_batches=max_batches,
            output_dir=output_dir,
        )

    def train(self, epochs):
        for epoch in range(1, epochs + 1):
            g_epoch, d_epoch, w_epoch = [], [], []
            for step, real_images in enumerate(self.dataset):
                for _ in range(self.trainer.n_critic):
                    d_loss, w_dist = self.trainer.critic_train_step(real_images)
                    d_epoch.append(float(d_loss.numpy()))
                    w_epoch.append(float(w_dist.numpy()))
                g_loss = self.trainer.generator_train_step()
                g_epoch.append(float(g_loss.numpy()))
                if self.max_batches and (step + 1) >= self.max_batches:
                    break
            self.history['g_loss'].append(float(np.mean(g_epoch)))
            self.history['d_loss'].append(float(np.mean(d_epoch)))
            self.history['wasserstein'].append(float(np.mean(w_epoch)))
            if self.sample_every and (epoch % self.sample_every == 0 or epoch == epochs):
                save_image_grid(
                    self.trainer.generator,
                    self.latent_dim,
                    self.sample_grid,
                    Path(self.output_dir) / f"samples_epoch_{epoch:03d}.png",
                )
            print(
                f"Epoch {epoch}/{epochs} | G loss: {self.history['g_loss'][-1]:.4f} "
                f"| D loss: {self.history['d_loss'][-1]:.4f} | W-dist: {self.history['wasserstein'][-1]:.4f}"
            )
        with open(self.output_dir / "history.json", "w") as f:
            json.dump(self.history, f, indent=2)
        with open(self.output_dir / "config.json", "w") as f:
            json.dump(self.config | {"epochs": epochs}, f, indent=2)
        save_image_grid(self.trainer.generator, self.latent_dim, self.sample_grid, self.output_dir / "final_samples.png")


In [ ]:

# === Quick Preview ===
train_dataset = get_mnist_dataset(batch_size=64)
preview_batch(train_dataset, n=16)


In [ ]:

# === Instantiate models & optimizers ===
LATENT_DIM = 128
BASE_CHANNELS = 96
BATCH_SIZE = 64
N_CRITIC = 3
GP_WEIGHT = 10.0
LR = 2e-4
BETAS = (0.0, 0.99)
SAMPLE_EVERY = 5
SAMPLE_GRID = 6
MAX_BATCHES = 0  # set >0 to limit batches per epoch for quick debugging
OUTPUT_DIR = "outputs/notebook_experiment"

G, D = create_models(latent_dim=LATENT_DIM, base_channels=BASE_CHANNELS)
notebook_trainer = NotebookTrainer(
    G,
    D,
    train_dataset,
    batch_size=BATCH_SIZE,
    latent_dim=LATENT_DIM,
    n_critic=N_CRITIC,
    gp_weight=GP_WEIGHT,
    lr=LR,
    beta1=BETAS[0],
    beta2=BETAS[1],
    output_dir=OUTPUT_DIR,
    sample_every=SAMPLE_EVERY,
    sample_grid=SAMPLE_GRID,
    max_batches=MAX_BATCHES,
    eager=False,
)
print(G.summary())
print(D.summary())


In [ ]:

# === Train ===
EPOCHS = 40
notebook_trainer.train(EPOCHS)
